# Gymnasium composition: measure attempts and tasks

## Point

Moving to the task level keeps episode measurements useful: you need their order to see adaptation and their sum to account for the whole task. Wrapper placement determines which level you measure.

## Objective

Build the inner environment first, pass it to `RepeatTaskEnv`, then add the Gymnasium wrappers that should see the whole task. Wrappers inside `RepeatTaskEnv` see each inner episode. Wrappers outside see the task, including the reset frame between episodes. This package supplies repetition and boundaries; Gymnasium supplies statistics, flattening, and vectorization.

The first cell records the same two-episode CartPole task at both levels:

- An inner `RecordEpisodeStatistics` writes each attempt's return and length to `info["episode"]`.
- An outer `RecordEpisodeStatistics` uses `stats_key="task"` to write task totals to `info["task"]` without colliding with the inner statistics.
- The task return is the sum of episode returns. Its length counts ten executed actions plus the one reset frame between attempts.

The random policy does not learn, and these short episodes both earn the same return. The measurements illustrate how to collect evidence; the improving episode-position scores in [notebook 00](00_episodes_to_tasks.ipynb) illustrate what adaptation can look like.


In [ ]:
import gymnasium as gym

from repeat_task_gym import RepeatTaskEnv


def make_task():
    base = gym.wrappers.RecordEpisodeStatistics(
        gym.make("CartPole-v1", max_episode_steps=5)
    )
    return RepeatTaskEnv(base, max_task_episodes=2)


env = gym.wrappers.RecordEpisodeStatistics(make_task(), stats_key="task")
episode_returns = []
episode_lengths = []
try:
    env.reset(seed=0)
    env.action_space.seed(0)
    while True:
        observation, _, terminated, truncated, info = env.step(env.action_space.sample())
        if observation["episode_terminated"] or observation["episode_truncated"]:
            episode_returns.append(info["episode"]["r"])
            episode_lengths.append(info["episode"]["l"])
        if terminated or truncated:
            print("Episode returns by attempt:", episode_returns)
            print("Episode lengths:", episode_lengths)
            print("Task statistics:", info["task"])
            assert episode_returns == [5.0, 5.0]
            assert episode_lengths == [5, 5]
            assert info["task"]["r"] == sum(episode_returns)
            assert info["task"]["l"] == sum(episode_lengths) + 1
            break
finally:
    env.close()


## Use the same tasks with other Gymnasium tools

`FlattenObservation` turns the augmented dictionary into one vector that still belongs to the flattened observation space. Both binary episode-boundary signals remain encoded in that vector.

`SyncVectorEnv` runs two separately constructed tasks side by side. Each has its own episode count and seed stream, and the batched observation keeps the `observation`, `episode_terminated`, and `episode_truncated` fields. Parallel execution is another dimension: each vector slot still contains its own sequence of attempts. An adapting agent would need separate task-local memory for each slot, cleared when that slot starts a new task.


In [ ]:
flat = gym.wrappers.FlattenObservation(make_task())
try:
    observation, _ = flat.reset(seed=0)
    assert observation in flat.observation_space
    print("Flat observation shape:", observation.shape)
finally:
    flat.close()

vector = gym.vector.SyncVectorEnv([make_task, make_task])
try:
    observations, _ = vector.reset(seed=0)
    vector.action_space.seed(0)
    assert observations["episode_terminated"].shape == (2,)
    assert observations["episode_truncated"].shape == (2,)
    for _ in range(30):
        observations, rewards, terminated, truncated, _ = vector.step(
            vector.action_space.sample()
        )
    print("Vector observation shape:", observations["observation"].shape)
finally:
    vector.close()


## Read episode returns alongside the task total

The episode returns describe successive attempts within the task. Their order shows the within-task learning progress we want to measure in meta-RL; their sum accounts for the whole task, including early exploration. If the task ends at first success, record the number of episodes it took to reach that success. With a fixed episode budget, later episodes can also show whether the agent reuses what it discovered.

Follow the task from its first episode through its last: keep useful information between episodes, observe how the attempts change, and finish at the task boundary. This succession of episodes is the unit we want to understand.

Return to [00 — From episodes to tasks](00_episodes_to_tasks.ipynb) for the concrete memory comparison, or the [README](../README.md) for the API reference.
